### Objetivo (Recall alto)

In [1]:
import polars as pl
from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold
)

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier

In [2]:
DATASET_FILE = Path('datasets/train.csv')

In [3]:
# Settings
TEST_SIZE = 0.2
EVAL_SIZE = 0.15
SEED = 42
MIN_FEATURES = 50

In [5]:
# load dataset
df = pl.read_csv(DATASET_FILE)

In [6]:
# organize features and targeta
TARGET = 'target'
FEATURES = [col for col in df.columns if col not in ('id', 'target')]

In [7]:
print(df.head(5))
print(df.shape)

shape: (5, 59)
┌─────┬────────┬───────────┬───────────────┬───┬────────────────┬────────────────┬────────────────┬────────────────┐
│ id  ┆ target ┆ ps_ind_01 ┆ ps_ind_02_cat ┆ … ┆ ps_calc_17_bin ┆ ps_calc_18_bin ┆ ps_calc_19_bin ┆ ps_calc_20_bin │
│ --- ┆ ---    ┆ ---       ┆ ---           ┆   ┆ ---            ┆ ---            ┆ ---            ┆ ---            │
│ i64 ┆ i64    ┆ i64       ┆ i64           ┆   ┆ i64            ┆ i64            ┆ i64            ┆ i64            │
╞═════╪════════╪═══════════╪═══════════════╪═══╪════════════════╪════════════════╪════════════════╪════════════════╡
│ 7   ┆ 0      ┆ 2         ┆ 2             ┆ … ┆ 1              ┆ 0              ┆ 0              ┆ 1              │
│ 9   ┆ 0      ┆ 1         ┆ 1             ┆ … ┆ 1              ┆ 0              ┆ 1              ┆ 0              │
│ 13  ┆ 0      ┆ 5         ┆ 4             ┆ … ┆ 1              ┆ 0              ┆ 1              ┆ 0              │
│ 16  ┆ 0      ┆ 0         ┆ 1             ┆ … ┆ 

In [8]:
print(df[TARGET].value_counts(normalize=True))

shape: (2, 2)
┌────────┬────────────┐
│ target ┆ proportion │
│ ---    ┆ ---        │
│ i64    ┆ f64        │
╞════════╪════════════╡
│ 1      ┆ 0.036448   │
│ 0      ┆ 0.963552   │
└────────┴────────────┘


In [9]:
# train test and eval dataset
df_train, df_test = train_test_split(
    df,
    stratify=df[TARGET],
    test_size=TEST_SIZE,
    random_state=SEED
)

df_train, df_eval = train_test_split(
    df_train,
    stratify=df_train[TARGET],
    test_size=EVAL_SIZE,
    random_state=SEED
)

In [10]:
estimator = SGDClassifier(
    loss="log_loss",
    random_state=SEED,
    max_iter=1000,
    tol=1e-3
)

pipeline_data = make_pipeline(
    PolynomialFeatures(interaction_only=False, include_bias=False),
    RFECV(
        estimator=estimator,
        step=20,
        min_features_to_select=MIN_FEATURES,
        cv=3
    )
)

In [11]:
pipeline_data

,steps,"[('polynomialfeatures', ...), ('rfecv', ...)]"
,transform_input,None
,memory,None
,verbose,False
,degree,2
,interaction_only,False
,include_bias,False
,order,'C'
,estimator,SGDClassifier...ndom_state=42)
,step,20
,min_features_to_select,50


In [12]:
X_train, y_train = df_train[FEATURES], df_train[TARGET]
X_eval, y_eval = df_eval[FEATURES], df_eval[TARGET]
X_test, y_test = df_test[FEATURES], df_test[TARGET]

In [ ]:
X_train = pipeline_data.fit_transform(X_train, y_train)

In [ ]:
X_eval = pipeline_data.transform(X_eval)
X_test = pipeline_data.transform(X_test)

In [ ]:
2+2

In [ ]:
2+2